# HotPotQA - ReAct + CoT (LangChain `create_agent` 버전)

CoT(Chain-of-Thought)를 ReAct 에이전트 내부에 통합한다.  
CoT는 별도 시스템이 아니라, ReAct의 매 Thought 단계에서 항상 동작한다.

| 방식 | 설명 |
|------|------|
| ReAct | 기본 프롬프트, temperature=0 |
| ReAct-CoT | CoT 강화 프롬프트, temperature=0 |
| ReAct-CoT-SC | ReAct-CoT를 N번 샘플링 + 다수결 투표 |

## 1. Setup

In [1]:
import os
import sys
import json
import random
import time

from dotenv import load_dotenv
load_dotenv("../.env")

from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

from tools import search, lookup, finish, wiki_state
from eval_utils import (
    load_hotpotqa, normalize_answer, f1_score, get_metrics, majority_vote
)

## 2. 데이터 로딩

In [2]:
data = load_hotpotqa("dev")
print(f"Loaded {len(data)} questions.")

Loaded 7405 questions.


## 3. 프롬프트 정의

기본 ReAct 프롬프트와 CoT 강화 프롬프트를 비교한다.  
CoT 강화 프롬프트는 매 Thought에서 단계적 추론을 명시적으로 요구한다.

In [3]:
# --- 기본 ReAct 프롬프트 ---
SYSTEM_PROMPT_BASE = """Thought, Action, Observation 단계를 번갈아가며 수행하여 질문 응답 과제를 해결하시오.

Thought는 현재 상황에 대해 추론하는 단계이며,
Action은 다음 세 가지 도구 중 하나를 사용해야 한다:

(1) search: Wikipedia에서 해당 entity를 검색하고, 존재할 경우 첫 번째 문단을 반환한다.
(2) lookup: 현재 문서에서 해당 keyword가 포함된 다음 문장을 반환한다.
(3) finish: 최종 답을 반환하고 과제를 종료한다. 답을 확정했을 때 반드시 이 도구를 호출하시오.

규칙:
- 각 단계에서 반드시 Thought를 먼저 수행한 뒤 도구를 호출하시오.
- 최대 7단계 이내에 답을 제출하시오.
- 최종 답은 간결하게 (단어 또는 짧은 구문) 제출하시오.
- 반드시 finish 도구를 호출하여 답을 제출하시오.
"""

# --- CoT 강화 ReAct 프롬프트 ---
SYSTEM_PROMPT_COT = """Thought, Action, Observation 단계를 번갈아가며 수행하여 질문 응답 과제를 해결하시오.

각 Thought 단계에서는 반드시 단계적으로 추론(chain-of-thought)하시오:
1. 현재까지 파악한 사실을 정리한다.
2. 질문에 답하기 위해 아직 필요한 정보를 식별한다.
3. 다음 행동을 선택한 근거를 설명한다.

Action은 다음 세 가지 도구 중 하나를 사용해야 한다:

(1) search: Wikipedia에서 해당 entity를 검색하고, 존재할 경우 첫 번째 문단을 반환한다.
(2) lookup: 현재 문서에서 해당 keyword가 포함된 다음 문장을 반환한다.
(3) finish: 최종 답을 반환하고 과제를 종료한다. 답을 확정했을 때 반드시 이 도구를 호출하시오.

규칙:
- 각 단계에서 반드시 Thought를 먼저 수행한 뒤 도구를 호출하시오.
- 최대 7단계 이내에 답을 제출하시오.
- 최종 답은 간결하게 (단어 또는 짧은 구문) 제출하시오.
- 반드시 finish 도구를 호출하여 답을 제출하시오.
"""

print(f"Base prompt: {len(SYSTEM_PROMPT_BASE)} chars")
print(f"CoT prompt:  {len(SYSTEM_PROMPT_COT)} chars")

Base prompt: 435 chars
CoT prompt:  534 chars


## 4. LLM 및 에이전트 생성

3개의 에이전트를 생성한다:
- `agent_base`: 기본 ReAct (temp=0)
- `agent_cot`: CoT 강화 ReAct (temp=0)
- `agent_cot_sc`: CoT 강화 ReAct + 샘플링 (temp=0.7, SC용)

In [4]:
llm_greedy = ChatOpenAI(model="gpt-4.1-mini", temperature=0, max_tokens=1000)
llm_sampling = ChatOpenAI(model="gpt-4.1-mini", temperature=0.7, max_tokens=1000)

tools = [search, lookup, finish]

# 1) 기본 ReAct
agent_base = create_agent(
    model=llm_greedy, tools=tools, system_prompt=SYSTEM_PROMPT_BASE,
)

# 2) ReAct-CoT (CoT 강화, 결정적)
agent_cot = create_agent(
    model=llm_greedy, tools=tools, system_prompt=SYSTEM_PROMPT_COT,
)

# 3) ReAct-CoT-SC (CoT 강화, 샘플링)
agent_cot_sc = create_agent(
    model=llm_sampling, tools=tools, system_prompt=SYSTEM_PROMPT_COT,
)

print("3 agents created.")

3 agents created.


## 5. 실행 함수 정의

In [5]:
def extract_answer(messages) -> str:
    """에이전트 응답에서 최종 답을 추출한다."""
    for msg in reversed(messages):
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            for tc in msg.tool_calls:
                if tc["name"] == "finish":
                    return tc["args"].get("answer", "")
    for msg in reversed(messages):
        content = msg.content if hasattr(msg, "content") else str(msg)
        if "Episode finished, answer:" in content:
            return content.split("Episode finished, answer:")[-1].strip()
    for msg in reversed(messages):
        if hasattr(msg, "content") and msg.type == "ai" and msg.content:
            return msg.content.strip()
    return ""


def run_agent(agent, question: str, to_print: bool = False) -> tuple[str, dict]:
    """에이전트를 실행하고 답과 메타 정보를 반환한다."""
    wiki_state.reset()
    result = agent.invoke({
        "messages": [{"role": "user", "content": f"Question: {question}"}]
    })
    messages = result["messages"]
    answer = extract_answer(messages)

    if to_print:
        print(f"Q: {question}")
        for msg in messages[1:]:
            role = msg.type if hasattr(msg, "type") else "unknown"
            content = msg.content if hasattr(msg, "content") else str(msg)
            if content:
                print(f"  [{role}] {content[:200]}")
        print(f"  => Answer: {answer}")

    return answer, {"answer": answer}


def run_agent_sc(
    agent, question: str, n_samples: int = 5, to_print: bool = False,
) -> tuple[str, dict]:
    """에이전트를 N번 실행하고 다수결 투표로 최종 답을 결정한다."""
    answers = []
    for i in range(n_samples):
        wiki_state.reset()
        try:
            result = agent.invoke({
                "messages": [{"role": "user", "content": f"Question: {question}"}]
            })
            answer = extract_answer(result["messages"])
        except Exception as e:
            answer = ""
        answers.append(answer)
        if to_print:
            print(f"  [Run {i+1}/{n_samples}] {answer}")

    # 빈 답 제거 후 투표
    valid_answers = [a for a in answers if a]
    if not valid_answers:
        return "", {"answer": "", "confidence": 0.0, "all_answers": answers}

    final_answer, confidence = majority_vote(valid_answers)

    if to_print:
        print(f"  Votes: {answers}")
        print(f"  => Final: {final_answer} (confidence: {confidence:.0%})")

    return final_answer, {
        "answer": final_answer,
        "confidence": confidence,
        "all_answers": answers,
    }


print("Run functions defined.")

Run functions defined.


## 6. 단일 질문 테스트 (3가지 방식 비교)

In [8]:
N_SC_SAMPLES = 5

question, gt_answer = data[0]
print(f"Question: {question}")
print(f"GT: {gt_answer}")
print("=" * 60)

# 1) ReAct (baseline)
print("\n[1] ReAct (baseline)")
pred1, _ = run_agent(agent_base, question, to_print=True)
print(f"EM: {normalize_answer(pred1) == normalize_answer(gt_answer)}")

print("\n" + "=" * 60)

# 2) ReAct-CoT
print("\n[2] ReAct-CoT")
pred2, _ = run_agent(agent_cot, question, to_print=True)
print(f"EM: {normalize_answer(pred2) == normalize_answer(gt_answer)}")

print("\n" + "=" * 60)

# 3) ReAct-CoT-SC
print(f"\n[3] ReAct-CoT-SC (n={N_SC_SAMPLES})")
pred3, info3 = run_agent_sc(agent_cot_sc, question, n_samples=N_SC_SAMPLES, to_print=True)
print(f"EM: {normalize_answer(pred3) == normalize_answer(gt_answer)}")

Question: Were Scott Derrickson and Ed Wood of the same nationality?
GT: yes

[1] ReAct (baseline)
Q: Were Scott Derrickson and Ed Wood of the same nationality?
  [ai] Thought: To answer whether Scott Derrickson and Ed Wood share the same nationality, I need to find out the nationality of each person.

Action: I will search for "Scott Derrickson" and "Ed Wood" on Wi
  [tool] Scott Derrickson (born July 16, 1966) is an American filmmaker. He is known for his work in the horror genre, directing films such as The Exorcism of Emily Rose (2005), Sinister (2012), The Black Phon
  [tool] Edward Davis Wood Jr. (October 10, 1924 – December 10, 1978) was an American filmmaker, actor, and pulp novelist.. In the 1950s, Wood directed several low-budget science fiction, crime and horror film
  [ai] Thought: Both Scott Derrickson and Ed Wood are described as American filmmakers in their respective Wikipedia summaries.

Action: I will now finish and provide the answer that they share the same nati
  [

KeyboardInterrupt: 

## 7. 배치 비교 평가

In [7]:
N_EVAL = 10
N_SC_SAMPLES = 5

idxs = list(range(len(data)))
random.Random(233).shuffle(idxs)

methods = {
    "ReAct": lambda q: run_agent(agent_base, q),
    "ReAct-CoT": lambda q: run_agent(agent_cot, q),
    "ReAct-CoT-SC": lambda q: run_agent_sc(agent_cot_sc, q, N_SC_SAMPLES),
}

all_results = {name: [] for name in methods}
old_time = time.time()

for i, idx in enumerate(idxs[:N_EVAL]):
    question, gt_answer = data[idx]
    print(f"\n[{i+1}/{N_EVAL}] Q: {question}")
    print(f"  GT: {gt_answer}")

    for method_name, method_fn in methods.items():
        try:
            pred, info = method_fn(question)
        except Exception as e:
            print(f"  {method_name}: ERROR - {e}")
            pred = ""

        metrics = get_metrics(pred, gt_answer)
        all_results[method_name].append(metrics)

        em_sum = sum(r["em"] for r in all_results[method_name])
        avg_em = em_sum / len(all_results[method_name])
        print(f"  {method_name:15s}: {pred[:50]:50s} | EM={avg_em:.3f} ({em_sum}/{len(all_results[method_name])})")

    elapsed = (time.time() - old_time) / (i + 1)
    print(f"  ({elapsed:.1f}s/q)")
    print("-" * 80)
    sys.stdout.flush()


[1/10] Q: What movie did actress Irene Jacob complete before the American action crime thriller film directed by Stuart Bird?
  GT: Beyond the Clouds
  ReAct          : Beyond the Clouds                                  | EM=1.000 (1/1)
  ReAct-CoT      : Beyond the Clouds                                  | EM=1.000 (1/1)


KeyboardInterrupt: 

## 8. 최종 결과 비교

In [ ]:
print(f"\n{'='*60}")
print(f"  Final Results ({N_EVAL} samples)")
print(f"  Agent: create_agent, SC n={N_SC_SAMPLES}")
print(f"{'='*60}")
print(f"{'Method':20s} {'EM':>8s} {'F1':>8s}")
print(f"{'-'*20} {'-'*8} {'-'*8}")

for method_name, results in all_results.items():
    n = len(results)
    if n == 0:
        continue
    total_em = sum(r["em"] for r in results)
    total_f1 = sum(r["f1"] for r in results)
    print(f"{method_name:20s} {total_em/n:8.3f} {total_f1/n:8.3f}")

print(f"{'='*60}")

## 9. 결과 저장

In [ ]:
os.makedirs("trajs", exist_ok=True)

summary = {}
for method_name, results in all_results.items():
    n = len(results)
    if n == 0:
        continue
    summary[method_name] = {
        "n_samples": n,
        "em": sum(r["em"] for r in results) / n,
        "f1": sum(r["f1"] for r in results) / n,
    }

with open("trajs/react_cot_langchain.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("Saved to trajs/react_cot_langchain.json")
print(json.dumps(summary, indent=2, ensure_ascii=False))